# Notebook 02: Motor de Generación de Datos Sintéticos

## Objetivo
Generar datasets ficticios parametrizados que simulen condiciones realistas de Sprints y proyectos ágiles.
Estos datos permiten probar la API, los dashboards y los modelos ML sin depender de datos reales,
garantizando la **portabilidad del prototipo** (RNF-06) y la facilidad de evaluación.

El motor analiza las distribuciones estadísticas reales del dataset TAWOS para replicar frecuencias
de tipos de tarea, rangos de esfuerzo y patrones de bloqueo de forma verosímil.

In [35]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)  # Semilla para reproducibilidad

# 1. CARGA DEL DATASET REAL (El "Molde")
df_real = pd.read_csv('../data/processed/dataset_entrenamiento_tawos.csv')

print(f"Dataset molde cargado: {df_real.shape[0]} tareas.")
print("Analizando distribuciones estadísticas reales para la simulación...")

# Extraemos las probabilidades reales de los tipos de tarea para que el simulador sea realista
prob_tipos = df_real['Issue_Type'].value_counts(normalize=True).to_dict()
tipos_tarea = list(prob_tipos.keys())
probabilidades = list(prob_tipos.values())
# 1b. DISTRIBUCIONES EMPÍRICAS DE TAWOS (para generar datos en-distribution)
# 1b. DISTRIBUCIONES EMPÍRICAS DE TAWOS (solo ACTIVE+CLOSED = tareas en sprint)
# Filtramos a tareas asignadas a sprints para que los datos sintéticos reflejen
# escenarios reales de análisis de sprint.
df_sprint = df_real[df_real['Sprint_State'].isin(['ACTIVE', 'CLOSED'])].copy()
story_point_dist = df_sprint['Story_Point'].value_counts(normalize=True)
total_effort_dist = df_sprint['Total_Effort_Minutes'].value_counts(normalize=True)
sprint_state_dist = df_sprint['Sprint_State'].value_counts(normalize=True)
print(f'Filtrado a {len(df_sprint)} tareas en sprint (ACTIVE+CLOSED).',
      f' Story_Point: {len(story_point_dist)} valores,',
      f' Esfuerzo: {len(total_effort_dist)} valores')


Dataset molde cargado: 142151 tareas.
Analizando distribuciones estadísticas reales para la simulación...
Filtrado a 32360 tareas en sprint (ACTIVE+CLOSED).  Story_Point: 38 valores,  Esfuerzo: 17816 valores


## 1. Motor de generación sintética (`Sprint Simulator`)
Se define la función `generar_sprint_sintetico`, que acepta tres niveles de caos configurables:
- **`perfecto`**: Sin bloqueos ni cambios, simula un sprint ideal.
- **`normal`**: Distribución realista con ~20% de probabilidad de cambios y bloqueos moderados.
- **`catastrófico`**: Alto caos, muchos bloqueos, cambios post-estimación (60%) y esfuerzo multiplicado ×2-3.5 para simular infraestimación severa.

Cada tarea incluye un título semántico contextualizado por tipo para alimentar al módulo LLM.

In [36]:
# --- BANCO DE TÍTULOS SINTÉTICOS (contexto semántico para el LLM) ---
TITULOS_POR_TIPO = {
    'Bug': [
        "Error de validación en formulario de alta de usuario",
        "Fallo en cálculo de totales en módulo de facturación",
        "NullPointerException al cargar el dashboard principal",
        "Timeout en llamada al endpoint de autenticación",
        "Datos duplicados al sincronizar con el servicio externo",
        "Excepción no controlada al exportar informe a PDF",
        "Sesión no invalidada correctamente al cerrar sesión",
    ],
    'Enhancement': [
        "Mejorar rendimiento de la consulta de histórico de pedidos",
        "Añadir paginación al listado de clientes",
        "Optimizar carga inicial del dashboard reduciendo llamadas API",
        "Reducir tiempo de respuesta del buscador global",
    ],
    'Task': [
        "Migrar configuración de entorno a variables de entorno",
        "Actualizar dependencias de seguridad del backend",
        "Refactorizar módulo de logging para cumplir estándar corporativo",
        "Configurar pipeline de CI/CD en el nuevo entorno de staging",
    ],
    'Story': [
        "Como usuario quiero filtrar el listado por fecha de creación",
        "Implementar exportación a CSV en el panel de informes",
        "Como admin quiero gestionar roles y permisos desde la UI",
        "Añadir notificaciones push al módulo de alertas",
    ],
    'Other': [
        "Revisión de documentación técnica del API REST",
        "Sesión de refinamiento de backlog Q3",
        "Investigación de alternativas para el módulo de reportes",
    ],
    'Epic': [
        "Rediseño del módulo de autenticación y gestión de sesiones",
        "Integración con sistema de pagos externo (Stripe)",
        "Migración de arquitectura monolítica a microservicios",
    ]
}

# 2. EL MOTOR DE GENERACIÓN SINTÉTICA
def generar_sprint_sintetico(nombre_proyecto, sprint_id, num_tareas, nivel_caos='normal', rango_retraso=None, factor_esfuerzo=None):
    """
    Genera un Sprint ficticio parametrizado para probar la API y los Dashboards.
    nivel_caos: 'perfecto' (0% riesgo), 'normal' (realidad), 'catastrofico' (muchos bloqueos)
    
    AHORA muestrea las distribuciones EMPÍRICAS de TAWOS para que las predicciones
    del modelo ML sean calibradas (mismo feature space que el entrenamiento).
    rango_retraso: tupla (min, max) para el ratio Resolution_Time / Total_Effort
    """
    sprint_data = []
    
    if nivel_caos == 'perfecto':
        prob_cambio = 0.01
        max_bloqueos = 0
        prob_cerrada = 0.6
        rango_retraso = rango_retraso or (0.8, 1.35)
    elif nivel_caos == 'catastrofico':
        prob_cambio = 0.60
        max_bloqueos = 5
        prob_cerrada = 0.15
        rango_retraso = rango_retraso or (2.0, 8.0)
    else:
        prob_cambio = 0.20
        max_bloqueos = 2
        prob_cerrada = 0.35
        rango_retraso = rango_retraso or (1.1, 4.0)

    base_date = pd.Timestamp('2025-01-06') + pd.Timedelta(days=(sprint_id - 1) * 14)

    for i in range(1, num_tareas + 1):
        # 1. Datos basicos
        issue_key = f'{nombre_proyecto[:3].upper()}-{sprint_id}0{i}'
        issue_type = np.random.choice(tipos_tarea, p=probabilidades)
        titulo = np.random.choice(TITULOS_POR_TIPO.get(issue_type, TITULOS_POR_TIPO['Other']))
        created_date = (base_date + pd.Timedelta(days=np.random.randint(0, 14))).strftime('%Y-%m-%d')
        
        # 2. Story_Point muestreado de distribucion empirica TAWOS (71% son 0)
        story_points = float(np.random.choice(
            story_point_dist.index,
            p=story_point_dist.values
        ))
        
        # 3. Total_Effort_Minutes muestreado de distribucion empirica TAWOS
        esfuerzo_base = float(np.random.choice(
            total_effort_dist.index,
            p=total_effort_dist.values
        ))
        
        if nivel_caos == 'catastrofico' and factor_esfuerzo is None:
            esfuerzo_base *= np.random.uniform(2.0, 3.5)
        elif factor_esfuerzo is not None:
            esfuerzo_base *= factor_esfuerzo
        
        # 4. Sprint_State muestreado de distribucion empirica TAWOS
        sprint_state = str(np.random.choice(
            sprint_state_dist.index,
            p=sprint_state_dist.values
        ))
        
        # 5. Simulacion de Riesgos
        title_changed = np.random.binomial(1, prob_cambio)
        desc_changed = np.random.binomial(1, min(prob_cambio, 0.05))  # ~0% en TAWOS
        sp_changed = np.random.binomial(1, prob_cambio / 2)
        bloqueos = np.random.randint(0, max_bloqueos + 1)
        
        # 6. Tiempo de resolucion con ratio controlado
        penalizacion_riesgo = 1.0 + (bloqueos * 0.3) + (desc_changed * 0.5)
        
        if np.random.rand() > 0.05:
            tiempo_resolucion = int(esfuerzo_base * np.random.uniform(*rango_retraso) * penalizacion_riesgo)
        else:
            tiempo_resolucion = int(esfuerzo_base * np.random.uniform(0.7, 1.0))
        
        # 7. In_Progress como fraccion de resolution_time
        in_progress = int(tiempo_resolucion * np.random.uniform(0.01, 0.5))

        # 8. Estado de la tarea
        status = 'Closed' if np.random.rand() < prob_cerrada else 'Open'

        sprint_data.append({
            'Issue_Key': issue_key,
            'Title': titulo,
            'Issue_Type': issue_type,
            'Project_ID': 999,
            'Project_Name': nombre_proyecto,
            'Sprint_ID': sprint_id,
            'Sprint_State': sprint_state,
            'Story_Point': story_points,
            'Total_Effort_Minutes': esfuerzo_base,
            'In_Progress_Minutes': float(in_progress),
            'Resolution_Time_Minutes': float(tiempo_resolucion),
            'Title_Changed_After_Estimation': title_changed,
            'Description_Changed_After_Estimation': desc_changed,
            'Story_Point_Changed_After_Estimation': sp_changed,
            'Blocker_Count': bloqueos,
            'Status': status,
            'Created_Date': created_date
        })
        
    return pd.DataFrame(sprint_data)

print('Motor de generacion sintetica inicializado.')


Motor de generacion sintetica inicializado.


## 2. Escenarios de prueba principales
Se generan tres Sprints aislados con niveles de caos diferentes para demostrar los estados del semáforo:
- **Sprint Normal** (30 tareas): Simulación de un sprint típico → semáforo Verde.
- **Sprint Catastrófico** (45 tareas): Esfuerzo infraestimado, bloqueos masivos → semáforo Rojo.
- **Sprint Perfecto** (50 tareas): Sprint ideal sin incidencias → semáforo Verde.

In [37]:
# 3. CREACIÓN DE ESCENARIOS DE PRUEBA

# Directorio de salida
out_dir = '../data/synthetic/'
os.makedirs(out_dir, exist_ok=True)

# Escenario A: Un Sprint normal de 30 tareas
df_sprint_normal = generar_sprint_sintetico("Proyecto Alpha", 1, 30, nivel_caos='normal')
df_sprint_normal.to_csv(f"{out_dir}sprint_normal.csv", index=False)

# Escenario B: El "Sprint Catastrófico" (Para probar las alertas rojas de la PMO)
df_sprint_caos = generar_sprint_sintetico("Proyecto Omega", 2, 45, nivel_caos='catastrofico')
df_sprint_caos.to_csv(f"{out_dir}sprint_catastrofico.csv", index=False)

# Escenario C: El "Sprint Perfecto" (Para engordar la clase minoritaria y que la IA aprenda)
df_sprint_perfecto = generar_sprint_sintetico("Proyecto Zen", 3, 50, nivel_caos='perfecto')
df_sprint_perfecto.to_csv(f"{out_dir}sprint_perfecto.csv", index=False)

# Escenario D: Sprint con riesgo moderado (esfuerzo ×1.5) para activar semáforo Amarillo
df_sprint_amarillo = generar_sprint_sintetico("Proyecto Gamma", 4, 40, nivel_caos='normal', factor_esfuerzo=1.5)
df_sprint_amarillo.to_csv(f"{out_dir}sprint_riesgo_moderado.csv", index=False)

print(f"✅ ¡Archivos generados con éxito en la carpeta '{out_dir}'!")
print(f" - Sprint Normal: {df_sprint_normal.shape[0]} tareas.")
print(f" - Sprint Catastrófico: {df_sprint_caos.shape[0]} tareas.")
print(f" - Sprint Perfecto: {df_sprint_perfecto.shape[0]} tareas.")
print(f" - Sprint Riesgo Moderado: {df_sprint_amarillo.shape[0]} tareas.")

✅ ¡Archivos generados con éxito en la carpeta '../data/synthetic/'!
 - Sprint Normal: 30 tareas.
 - Sprint Catastrófico: 45 tareas.
 - Sprint Perfecto: 50 tareas.
 - Sprint Riesgo Moderado: 40 tareas.


## 3. Proyecto completo (Vista PMO)
Se simula la evolución del **Proyecto Delta** a lo largo de 4 Sprints consecutivos:
1. Sprint 1: Arranque normal.
2. Sprint 2: Comienzan a aparecer desviaciones.
3. Sprint 3: Crisis del proyecto (catastrófico) — ideal para que salten las alarmas.
4. Sprint 4: Se aplican medidas correctivas y se estabiliza.

Este archivo maestro permite probar la vista de **Alcance Proyecto** y las tendencias temporales del dashboard.

In [38]:
# 4. CREACIÓN DE UN PROYECTO COMPLETO (Para la vista de la PMO)

sprints_proyecto = []

# Simulamos la evolución del "Proyecto Delta" a lo largo de 4 Sprints
# Sprint 1: Arranque normal
sprints_proyecto.append(generar_sprint_sintetico("Proyecto Delta", 1, 30, nivel_caos='normal'))

# Sprint 2: Empieza a torcerse un poco
sprints_proyecto.append(generar_sprint_sintetico("Proyecto Delta", 2, 35, nivel_caos='normal'))

# Sprint 3: La crisis del proyecto (ideal para que salten las alarmas de la PMO)
sprints_proyecto.append(generar_sprint_sintetico("Proyecto Delta", 3, 40, nivel_caos='catastrofico'))

# Sprint 4: Se aplican medidas y se estabiliza
sprints_proyecto.append(generar_sprint_sintetico("Proyecto Delta", 4, 25, nivel_caos='perfecto'))

# Unimos todos los sprints en un único archivo maestro del proyecto
df_proyecto_completo = pd.concat(sprints_proyecto, ignore_index=True)
df_proyecto_completo.to_csv(f"{out_dir}proyecto_completo.csv", index=False)

print(
    f"✅ ¡Proyecto completo generado! "
    f"'proyecto_completo.csv' con {df_proyecto_completo.shape[0]} "
    f"tareas en total distribuidas en 4 Sprints."
)

✅ ¡Proyecto completo generado! 'proyecto_completo.csv' con 130 tareas en total distribuidas en 4 Sprints.


In [39]:
# 5. ESCENARIOS ADICIONALES PARA TESTING EXHAUSTIVO
out_dir = '../data/synthetic/'

# 5a. Sprint Perfecto Sin Retraso (0% retrasos por definicion de Target_Retraso)
# Resolution_Time <= 1.40 * Total_Effort para TODAS las tareas
df_perfecto_real = generar_sprint_sintetico("Perfecto Real", 10, 50, nivel_caos='perfecto', rango_retraso=(0.6, 1.35))
df_perfecto_real.to_csv(f"{out_dir}sprint_perfecto_sin_retraso.csv", index=False)
ratio_perfecto = df_perfecto_real['Resolution_Time_Minutes'] / df_perfecto_real['Total_Effort_Minutes']
retrasos_perfecto = (ratio_perfecto > 1.40).mean() * 100
print(f"5a. Sprint Perfecto Sin Retraso: {df_perfecto_real.shape[0]} tareas, {retrasos_perfecto:.0f}% con retraso")

# 5b. Sprint Retraso Moderado (~25% de tareas con retraso)
df_retraso_bajo = generar_sprint_sintetico("Retraso Bajo", 11, 40, nivel_caos='normal', rango_retraso=(0.8, 1.7))
df_retraso_bajo.to_csv(f"{out_dir}sprint_retraso_moderado.csv", index=False)
ratio_bajo = df_retraso_bajo['Resolution_Time_Minutes'] / df_retraso_bajo['Total_Effort_Minutes']
retrasos_bajo = (ratio_bajo > 1.40).mean() * 100
print(f"5b. Sprint Retraso Moderado: {df_retraso_bajo.shape[0]} tareas, {retrasos_bajo:.0f}% con retraso")

# 5c. Sprint Retraso Critico (~80% de tareas con retraso)
df_retraso_alto = generar_sprint_sintetico("Retraso Alto", 12, 45, nivel_caos='normal', rango_retraso=(1.3, 2.8))
df_retraso_alto.to_csv(f"{out_dir}sprint_retraso_critico.csv", index=False)
ratio_alto = df_retraso_alto['Resolution_Time_Minutes'] / df_retraso_alto['Total_Effort_Minutes']
retrasos_alto = (ratio_alto > 1.40).mean() * 100
print(f"5c. Sprint Retraso Critico: {df_retraso_alto.shape[0]} tareas, {retrasos_alto:.0f}% con retraso")

# 5d. Sprint Solo Bloqueos (sin cambios post-estimacion)
# Forzamos bloqueos altos pero sin cambios, para aislar su efecto
def generar_sprint_solo_bloqueos(nombre, sid, n):
    """Genera sprint con bloqueos pero sin cambios post-estimacion."""
    df = generar_sprint_sintetico(nombre, sid, n, nivel_caos='catastrofico', rango_retraso=(1.1, 2.5))
    df['Title_Changed_After_Estimation'] = 0
    df['Description_Changed_After_Estimation'] = 0
    df['Story_Point_Changed_After_Estimation'] = 0
    return df

df_solo_bloqueos = generar_sprint_solo_bloqueos("Solo Bloqueos", 13, 35)
df_solo_bloqueos.to_csv(f"{out_dir}sprint_solo_bloqueos.csv", index=False)
print(f"5d. Sprint Solo Bloqueos: {df_solo_bloqueos.shape[0]} tareas, "
      f"bloqueos max={df_solo_bloqueos['Blocker_Count'].max()}")

# 5e. Sprint Solo Cambios (sin bloqueos)
def generar_sprint_solo_cambios(nombre, sid, n):
    """Genera sprint con cambios post-estimacion pero sin bloqueos."""
    df = generar_sprint_sintetico(nombre, sid, n, nivel_caos='normal', rango_retraso=(1.1, 2.5))
    df['Blocker_Count'] = 0
    return df

df_solo_cambios = generar_sprint_solo_cambios("Solo Cambios", 14, 35)
df_solo_cambios.to_csv(f"{out_dir}sprint_solo_cambios.csv", index=False)
print(f"5e. Sprint Solo Cambios: {df_solo_cambios.shape[0]} tareas, "
      f"cambios max={df_solo_cambios[['Title_Changed_After_Estimation','Description_Changed_After_Estimation','Story_Point_Changed_After_Estimation']].sum(axis=1).max()}")

# 5f. Archivo multi-proyecto (debe fallar validacion de alcance Proyecto)
df_proy_a = generar_sprint_sintetico("Proyecto A", 1, 20, nivel_caos='normal')
df_proy_b = generar_sprint_sintetico("Proyecto B", 1, 20, nivel_caos='normal')
df_multi_proy = pd.concat([df_proy_a, df_proy_b], ignore_index=True)
df_multi_proy.to_csv(f"{out_dir}varios_proyectos.csv", index=False)
print(f"5f. Multi-Proyecto: {df_multi_proy.shape[0]} tareas, "
      f"proyectos={df_multi_proy['Project_Name'].nunique()}")

# 5g. Archivo multi-sprint (debe fallar validacion de alcance Sprint)
df_s1 = generar_sprint_sintetico("Multi Sprint", 1, 15, nivel_caos='normal')
df_s2 = generar_sprint_sintetico("Multi Sprint", 2, 15, nivel_caos='normal')
df_multi_sprint = pd.concat([df_s1, df_s2], ignore_index=True)
df_multi_sprint.to_csv(f"{out_dir}varios_sprints.csv", index=False)
print(f"5g. Multi-Sprint: {df_multi_sprint.shape[0]} tareas, "
      f"sprints={df_multi_sprint['Sprint_ID'].nunique()}")

# 5h. Proyecto que mejora (estabilizacion progresiva)
sprints_mejora = [
    generar_sprint_sintetico("Proyecto Mejora Progresiva", 1, 30, nivel_caos='catastrofico'),
    generar_sprint_sintetico("Proyecto Mejora Progresiva", 2, 30, nivel_caos='normal', rango_retraso=(1.2, 2.5)),
    generar_sprint_sintetico("Proyecto Mejora Progresiva", 3, 25, nivel_caos='normal'),
    generar_sprint_sintetico("Proyecto Mejora Progresiva", 4, 20, nivel_caos='perfecto', rango_retraso=(0.6, 1.35)),
]
df_mejora = pd.concat(sprints_mejora, ignore_index=True)
df_mejora.to_csv(f"{out_dir}proyecto_mejora_progresiva.csv", index=False)
print(f"5h. Proyecto Mejora Progresiva: {df_mejora.shape[0]} tareas en 4 sprints")

# 5i. Proyecto que empeora (crisis progresiva)
sprints_empeora = [
    generar_sprint_sintetico("Proyecto Crisis Progresiva", 1, 20, nivel_caos='perfecto', rango_retraso=(0.6, 1.35)),
    generar_sprint_sintetico("Proyecto Crisis Progresiva", 2, 25, nivel_caos='normal'),
    generar_sprint_sintetico("Proyecto Crisis Progresiva", 3, 30, nivel_caos='normal', rango_retraso=(1.2, 2.8)),
    generar_sprint_sintetico("Proyecto Crisis Progresiva", 4, 35, nivel_caos='catastrofico'),
]
df_empeora = pd.concat(sprints_empeora, ignore_index=True)
df_empeora.to_csv(f"{out_dir}proyecto_crisis_progresiva.csv", index=False)
print(f"5i. Proyecto Crisis Progresiva: {df_empeora.shape[0]} tareas en 4 sprints")

print(f"\nTotal: 9 nuevos datasets generados en '{out_dir}'")


5a. Sprint Perfecto Sin Retraso: 50 tareas, 2% con retraso
5b. Sprint Retraso Moderado: 40 tareas, 62% con retraso
5c. Sprint Retraso Critico: 45 tareas, 89% con retraso
5d. Sprint Solo Bloqueos: 35 tareas, bloqueos max=4
5e. Sprint Solo Cambios: 35 tareas, cambios max=1
5f. Multi-Proyecto: 40 tareas, proyectos=2
5g. Multi-Sprint: 30 tareas, sprints=2
5h. Proyecto Mejora Progresiva: 105 tareas en 4 sprints
5i. Proyecto Crisis Progresiva: 110 tareas en 4 sprints

Total: 9 nuevos datasets generados en '../data/synthetic/'


## Guía rápida de datasets para la defensa del prototipo

| Dataset | Alcance | Semáforo esperado | Qué demuestra |
|---|---|---|---|
| `sprint_perfecto.csv` | Sprint | Verde | Todo bajo control, sin alertas |
| `sprint_normal.csv` | Sprint | Verde | Funcionamiento estándar, KPIs normales |
| `sprint_riesgo_moderado.csv` | Sprint | Amarillo | Riesgo moderado, alertas intermedias |
| `sprint_catastrofico.csv` | Sprint | Rojo | Alertas máximas, recomendaciones urgentes |
| `sprint_solo_bloqueos.csv` | Sprint | Rojo | Impacto aislado de bloqueos |
| `sprint_retraso_critico.csv` | Sprint | Verde/Rojo | Alerta de retraso activada con riesgo bajo |
| `proyecto_completo.csv` | Proyecto | Amarillo/Rojo (S3) | Evolución de 4 sprints, tendencias temporales |
| `proyecto_mejora_progresiva.csv` | Proyecto | Rojo→Verde | Tendencia descendente de riesgo |
| `proyecto_crisis_progresiva.csv` | Proyecto | Verde→Rojo | Tendencia ascendente de riesgo |